# Análise interativa - Solução 2 U-Net leve

**Autor:** Manoel Furtado  
**Objetivo:** entender passo a passo a preparação de máscaras e o treino opcional da U-Net leve implementada em `solucao_2_unet_leve.py`.

Este notebook foi feito para estudo. Ele mostra como:

1. localizar pares imagem/label do Desafio 2;
2. converter polígonos YOLO em máscaras binárias;
3. visualizar a divisão treino/validação;
4. inspecionar o formato de entrada esperado pela U-Net;
5. testar parâmetros como `size`, `batch`, `lr` e `epochs`;
6. executar um treino curto ou carregar pesos já treinados para inferência.

A ideia é alterar as variáveis nas células e executar novamente para observar o impacto.

## 1. Importações e caminhos

A célula abaixo localiza automaticamente a pasta da solução, adiciona o script ao `sys.path` e aponta para o dataset do Desafio 2.

In [ ]:
import random
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

SCRIPT_NAME = 'solucao_2_unet_leve.py'

def find_solution_dir(script_name):
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / script_name).exists():
            return candidate
    matches = list(start.rglob(script_name))
    if not matches:
        raise FileNotFoundError(f'Não encontrei {script_name} a partir de {start}')
    return matches[0].parent

SOLUTION_DIR = find_solution_dir(SCRIPT_NAME)
DESAFIO_DIR = SOLUTION_DIR.parents[1]
DATA_DIR = DESAFIO_DIR / 'data'
OUTPUT_DIR = SOLUTION_DIR / 'dataset_unet_masks'

sys.path.insert(0, str(SOLUTION_DIR))

from solucao_2_unet_leve import label_to_mask, paired_items, parse_args, prepare_masks, train_tiny_unet

print('Solução:', SOLUTION_DIR)
print('Dados   :', DATA_DIR)
print('Saída   :', OUTPUT_DIR)

## 2. Conferência dos pares imagem/label

A U-Net precisa de uma máscara para cada imagem. Por isso começamos verificando quais imagens possuem um `.txt` correspondente na pasta `labels`.

In [ ]:
pairs = list(paired_items(DATA_DIR))
print('Pares imagem/label encontrados:', len(pairs))

for image_path, label_path in pairs[:5]:
    print(image_path.name, '->', label_path.name)

## 3. Escolha de uma imagem e conversão do label

Troque `image_index` para observar outro exemplo. A máscara branca representa pixels anotados como fissura.

In [ ]:
image_index = 0

image_path, label_path = pairs[image_index]
image_bgr = cv2.imread(str(image_path))
if image_bgr is None:
    raise ValueError(f'Não consegui ler {image_path}')

height, width = image_bgr.shape[:2]
mask = label_to_mask(cv2, np, label_path, width, height)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Imagem: {image_path.name}')
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Máscara binária')
axes[1].axis('off')

overlay = image_bgr.copy()
overlay[mask > 0] = (0, 0, 255)
blend = cv2.addWeighted(image_bgr, 0.75, overlay, 0.25, 0)
axes[2].imshow(cv2.cvtColor(blend, cv2.COLOR_BGR2RGB))
axes[2].set_title('Sobreposição')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print('Dimensão da imagem:', image_bgr.shape)
print('Pixels positivos na máscara:', int((mask > 0).sum()))
print('Percentual de fissura:', round(100 * float((mask > 0).mean()), 4), '%')

## 4. Como o split treino/validação é definido

A função `prepare_masks` embaralha os pares com uma seed fixa e separa uma fração para validação. Aqui simulamos a divisão sem copiar arquivos.

In [ ]:
seed = 42
val_fraction = 0.15

items = pairs.copy()
random.seed(seed)
random.shuffle(items)

val_count = int(len(items) * val_fraction)
val_items = items[:val_count]
train_items = items[val_count:]

print('Seed:', seed)
print('Fração de validação:', val_fraction)
print('Treino:', len(train_items))
print('Validação:', len(val_items))
print('Primeira imagem de validação:', val_items[0][0].name if val_items else 'sem validação')

## 5. Preparação das máscaras no disco

Por padrão a célula não executa a preparação para evitar sobrescrever uma pasta pronta. Mude `execute_prepare` para `True` se quiser gerar `dataset_unet_masks` novamente.

In [ ]:
execute_prepare = False

args_prepare = parse_args([
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--prepare',
    '--seed', str(seed),
    '--val-fraction', str(val_fraction),
])

if execute_prepare:
    prepared_dir = prepare_masks(args_prepare)
    print('Dataset preparado em:', prepared_dir)
else:
    print('Preparação desativada. Para executar, mude execute_prepare = True.')
    print('Comando equivalente:')
    print(f'python {SOLUTION_DIR / SCRIPT_NAME} --prepare --data-dir {DATA_DIR} --output-dir {OUTPUT_DIR}')

## 6. Inspeção do dataset preparado

Depois que `dataset_unet_masks` existir, esta célula mostra um par imagem/máscara já salvo na estrutura usada pelo treino.

In [ ]:
split = 'train'
prepared_images = sorted((OUTPUT_DIR / split / 'images').glob('*'))

if not prepared_images:
    print('Dataset preparado não encontrado. Execute a célula anterior com execute_prepare = True.')
else:
    prepared_index = 0
    prepared_image_path = prepared_images[prepared_index]
    prepared_mask_path = OUTPUT_DIR / split / 'masks' / f'{prepared_image_path.stem}.png'

    prepared_image = cv2.imread(str(prepared_image_path))
    prepared_mask = cv2.imread(str(prepared_mask_path), cv2.IMREAD_GRAYSCALE)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(prepared_image, cv2.COLOR_BGR2RGB))
    axes[0].set_title(prepared_image_path.name)
    axes[0].axis('off')
    axes[1].imshow(prepared_mask, cmap='gray')
    axes[1].set_title(prepared_mask_path.name)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    print('Imagem:', prepared_image.shape)
    print('Máscara:', prepared_mask.shape, 'valores únicos:', np.unique(prepared_mask)[:10])

## 7. O que entra na rede

A U-Net recebe tensores no formato `N x C x H x W`. A célula abaixo replica a arquitetura do script para mostrar o tamanho da saída antes do treino.

In [ ]:
try:
    import torch
    import torch.nn as nn
except ImportError:
    torch = None
    print('PyTorch não está disponível. Ative o ambiente conda vc_01 para esta seção.')

if torch is not None:
    class Block(nn.Module):
        def __init__(self, in_channels, out_channels):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, 3, padding=1),
                nn.ReLU(inplace=True),
            )
        def forward(self, x):
            return self.net(x)

    class TinyUNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.down1 = Block(3, 16)
            self.pool1 = nn.MaxPool2d(2)
            self.down2 = Block(16, 32)
            self.pool2 = nn.MaxPool2d(2)
            self.bridge = Block(32, 64)
            self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
            self.dec2 = Block(64, 32)
            self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
            self.dec1 = Block(32, 16)
            self.out = nn.Conv2d(16, 1, 1)
        def forward(self, x):
            d1 = self.down1(x)
            d2 = self.down2(self.pool1(d1))
            b = self.bridge(self.pool2(d2))
            u2 = self.up2(b)
            u2 = torch.cat([u2, d2], dim=1)
            u2 = self.dec2(u2)
            u1 = self.up1(u2)
            u1 = torch.cat([u1, d1], dim=1)
            return self.out(self.dec1(u1))

    size = 384
    model = TinyUNet()
    dummy = torch.zeros(1, 3, size, size)
    with torch.no_grad():
        logits = model(dummy)
    print('Entrada :', tuple(dummy.shape))
    print('Saída   :', tuple(logits.shape))
    print('Parâmetros treináveis:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 8. Experimento rápido de parâmetros

`size` controla o custo por imagem. Aumentar o tamanho melhora detalhes, mas aumenta memória e tempo de treino. Esta célula ajuda a comparar configurações antes de treinar.

In [ ]:
sizes = [256, 384, 512]
batches = [2, 4, 8]

print('Custo relativo aproximado em pixels por batch:')
for size_option in sizes:
    for batch_option in batches:
        pixels = batch_option * size_option * size_option
        print(f'size={size_option:3d} batch={batch_option:2d} -> {pixels / 1_000_000:.2f} MPixels')

print('\nSugestão didática: comece com size=256 e epochs=2 para testar o fluxo.')

## 9. Treino curto opcional

O treino completo pode demorar. Por padrão esta célula fica desligada. Para um teste rápido, prepare o dataset e rode poucas épocas.

In [ ]:
execute_training = False

args_train = parse_args([
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--train',
    '--size', '256',
    '--epochs', '2',
    '--batch', '2',
    '--lr', '0.001',
])

if execute_training:
    if not OUTPUT_DIR.exists():
        prepare_masks(args_train)
    train_tiny_unet(args_train, OUTPUT_DIR)
else:
    print('Treino desativado. Mude execute_training = True para rodar.')

## 10. Inferência com pesos salvos

Se `tiny_unet_fissuras.pt` existir, esta célula carrega os pesos e mostra a máscara predita em uma imagem.

In [ ]:
weights_path = OUTPUT_DIR / 'tiny_unet_fissuras.pt'
inference_size = 384
threshold = 0.50

if torch is None:
    print('PyTorch não disponível.')
elif not weights_path.exists():
    print('Pesos não encontrados:', weights_path)
else:
    model = TinyUNet()
    model.load_state_dict(torch.load(weights_path, map_location='cpu'))
    model.eval()

    image_bgr = cv2.imread(str(image_path))
    resized = cv2.resize(image_bgr, (inference_size, inference_size))
    image_rgb = resized[:, :, ::-1].astype(np.float32) / 255.0
    tensor = torch.tensor(np.transpose(image_rgb, (2, 0, 1))[None, :, :, :])

    with torch.no_grad():
        prob = torch.sigmoid(model(tensor))[0, 0].numpy()
    pred_mask = (prob >= threshold).astype(np.uint8) * 255

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Imagem redimensionada')
    axes[0].axis('off')
    axes[1].imshow(prob, cmap='magma')
    axes[1].set_title('Probabilidade de fissura')
    axes[1].axis('off')
    axes[2].imshow(pred_mask, cmap='gray')
    axes[2].set_title(f'Máscara com threshold={threshold}')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

## 11. Dicas de estudo

- Se a máscara real aparece muito fina, revise se o label original é poligonal e não apenas uma linha.
- Se o treino parece instável, reduza `lr` para `0.0003` ou use mais épocas.
- Se falta memória, reduza `size` ou `batch`.
- Para comparar com YOLO, use a mesma divisão ou registre a seed usada no relatório.
- O modelo salvo é apenas `state_dict`; a arquitetura precisa estar definida para carregar os pesos.